<a href="https://colab.research.google.com/github/yeatescp01/INFO648/blob/main/Lesson_07/lesson_lasso_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3: Lasso Regression
## Feature Scaling and Automatic Feature Selection

**What this notebook does:**

1. Reuses the Lesson 2 house-price recipe and **adds 5 junk feature columns** that have nothing to do with price.
2. Shows what **`StandardScaler`** actually does to the numeric features.
3. Fits plain `LinearRegression` and shows that *every* feature (including the junk) gets a nonzero coefficient.
4. Fits `Lasso` with a hand-picked alpha and shows the junk coefficients are pushed exactly to **zero**.
5. Sweeps alpha and plots the **coefficient paths** so students can watch features drop out as the penalty grows.
6. Uses **`LassoCV`** to pick alpha automatically with 5-fold cross-validation, and compares the test MAE.

Companion notes: `lesson3_lasso.pdf`.

## 0. Setup

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error



## 1. Lesson 2's house-price recipe + 5 junk columns

Same houses as before: `sqft`, `bedrooms`, `age`, `neighborhood`. Now we also pretend a data analyst gave us 5 extra columns that have **nothing to do with price**:

- `lucky_number`: a random integer between 1 and 100
- `paperclips_sold`: a random count
- `favorite_color_hue`: a random degree on the color wheel
- `moon_phase`: between 0 and 1
- `temperature_F`: room temperature when the home was listed

We'd love a model that ignores these. That's exactly the job Lasso is good at.

We also use a **smaller training set than Lesson 2** (80 homes instead of 200). Lasso matters most when data is limited and features are many --- which is exactly when plain linear regression starts to overfit.

In [2]:
homes=pd.read_csv('homes_2.csv')

In [3]:
homes.head()

,Unnamed: 0,sqft,bedrooms,age,neighborhood,lucky_number,paperclips_sold,favorite_color_hue,moon_phase,temperature_F,price_k
0,0,2520,5,52,Suburb,51,20,203.2,0.202,70.8,626.7
1,1,1528,4,27,Suburb,19,25,174.4,0.938,77.3,442.6
2,2,911,3,18,Suburb,26,25,323.6,0.095,75.8,334.0
3,3,845,3,34,Rural,97,37,31.0,0.005,70.7,260.2
4,4,2996,3,69,Suburb,5,25,250.6,0.323,79.1,677.0


## 2. Train/test split (same as before)
Seperate the features and the target and split the data into test and train.

In [4]:
feature_cols = homes.drop('price_k',axis=1)
X = feature_cols
y = homes['price_k']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f'training homes = {len(X_train)},  test homes = {len(X_test)}')


training homes = 60,  test homes = 20


## 3. What is MinMaxScaler actually doing?
Because lasso regression uses $α$ as a set penalty to minimize the number of coefficants we have to scale our data. There are a lot of ways of doing this - one is minmaxscaler.

### Lasso Regression Objective Function

Lasso regression finds the optimal coefficients by solving the following minimization problem:

Before we wrap it inside a Pipeline, let's see scaling in action so it isn't magic.

**Each numeric feature is rewritten as `(value - min) / (max-min)`.** After scaling, every numeric column has a max of 1 and a min of 0.

Example: sqrft: 2520,1528,911,845.
min=845
max=2520

$$\frac{1528-845}{2520-845}=0.408$$

## 4. Pipeline: same as Lesson 2, with MinMaxScaler added

In real code we don't call the scaler by hand. We put it on the numeric branch of the ColumnTransformer so the Pipeline takes care of fitting it on the training data only.

In [5]:
categorical=['neighborhood']
numeric=['sqft',	'bedrooms',	'age',	'lucky_number',	'paperclips_sold',
              'favorite_color_hue',	'moon_phase',	'temperature_F']

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat',
         OneHotEncoder(handle_unknown='ignore'),
         categorical),
        ('num', MinMaxScaler(), numeric),
    ]
)



Every junk feature got a nonzero coefficient. Plain linear regression has no way to say "this feature doesn't matter"; it always fits something. And with only 60 training homes, the test MAE is notably worse than the training MAE --- classic overfitting.

## 6. Lasso with a hand-picked alpha

Now swap in `Lasso(alpha=1.0)` and refit. The Pipeline doesn't change.

In [7]:
lasso_pipe = Pipeline([
    ('prep',  preprocessor),
    ('lasso', Lasso(alpha=1, max_iter=10000)),
])

lasso_pipe.fit(X_train,y_train)


# Tag junk features for easier reading


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['neighborhood']),
                                                 ('num', MinMaxScaler(),
                                                  ['sqft', 'bedrooms', 'age',
                                                   'lucky_number',
                                                   'paperclips_sold',
                                                   'favorite_color_hue',
                                                   'moon_phase',
                                                   'temperature_F'])])),
                ('lasso', Lasso(alpha=1, max_iter=10000))])

In [8]:
y_pred=lasso_pipe.predict(X_test)

In [11]:
mae=mean_absolute_error(y_test,y_pred)
rmse=root_mean_squared_error(y_test,y_pred)
print(mae,rmse)

30.458359307034623 40.47957611163361


In [12]:
lasso_model = lasso_pipe.named_steps['lasso']
feature_names = lasso_pipe.named_steps['prep'].get_feature_names_out()
coefficients = pd.Series(lasso_model.coef_, index=feature_names)
print("Lasso Model Coefficients:")
print(coefficients)

Lasso Model Coefficients:
cat__neighborhood_Downtown     14.655136
cat__neighborhood_Rural       -29.511274
cat__neighborhood_Suburb        0.000000
num__sqft                     475.145988
num__bedrooms                  78.104793
num__age                      -18.508213
num__lucky_number             -22.968950
num__paperclips_sold           -0.000000
num__favorite_color_hue         3.282071
num__moon_phase                -0.000000
num__temperature_F              0.000000
dtype: float64


Look at the `Lasso(alpha=1.0)` column above. The junk features are at (or very near) **zero**, while the real signals (`sqft`, `bedrooms`, `age`, the neighborhood dummies) are kept. The Lasso has performed feature selection automatically.

## 7. Coefficient paths: how the coefficients evolve with alpha

Let's sweep alpha from very small (almost no penalty) to large (very strong penalty) and watch each feature's coefficient. Real features hold on; junk features die off quickly.

Reading the plot:

- At very small alpha (left), all coefficients look like the unpenalized linear regression.
- As alpha grows, coefficients shrink toward zero. **Junk features cross zero first.**
- At very large alpha (right), only the strongest signals (or none) remain.
- The right alpha is somewhere in the middle. We don't want to eyeball it --- we use cross-validation.